# **Naive Bayes Classifier**
## **SMS Spam Collection & BBC News Dataset**

## *Introduction:*

This project explores how we can teach a computer to understand and classify text using a simple yet powerful technique — the Naive Bayes algorithm.
It works by using probabilities of words and categories to make predictions, assuming that all words in a document are independent of each other.

We’ll apply this approach to two real-world datasets:

1) BBC News Dataset – A collection of news articles grouped into five categories: business, entertainment, politics, sport, and tech.

2) SMS Spam Dataset – A smaller dataset containing text messages labeled as either ham (not spam) or spam.

By building everything from scratch, we’ll gain a clear understanding of how text classification actually works under the hood — from cleaning the raw text, to computing probabilities, to making predictions and analyzing results.

In short, our goals are to:

1. Load and clean text data.

2. Implement a Naive Bayes model manually.

3. Evaluate its performance using accuracy and confusion matrices.

4. Identify the most indicative words for each class.

In [2]:
import zipfile
import os
import pandas as pd
import numpy as np
import string
from collections import Counter
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS
import stopwordsiso as stopwords
stopwords_iso=stopwords.stopwords("en")

# function to split the data into training and test sets

def train_test_split_df(df, test_frac=0.2, label_col='Label', random_seed=42):
    np.random.seed(random_seed)
    train_idx = []
    test_idx = []

    for label in df[label_col].unique():
        idx = df[df[label_col] == label].index.values
        np.random.shuffle(idx)
        test_size = int(len(idx) * test_frac)
        test_idx.extend(idx[:test_size])
        train_idx.extend(idx[test_size:])

    return df.loc[train_idx].reset_index(drop=True), df.loc[test_idx].reset_index(drop=True)

#function to preprocess the data

stop_words = set(ENGLISH_STOP_WORDS)
stop_words.update( {
    'a', 'about', 'above', 'after', 'again', 'against', 'all', 'am', 'an', 'and',
    'any', 'are', 'aren’t', 'as', 'at', 'be', 'because', 'been', 'before', 'being',
    'below', 'between', 'both', 'but', 'by', 'can', 'can’t', 'cannot', 'could',
    'couldn’t', 'did', 'didn’t', 'do', 'does', 'doesn’t', 'doing', 'don’t', 'down',
    'during', 'each', 'few', 'for', 'from', 'further', 'had', 'hadn’t', 'has',
    'hasn’t', 'have', 'haven’t', 'having', 'he', 'he’d', 'he’ll', 'he’s', 'her',
    'here', 'here’s', 'hers', 'herself', 'him', 'himself', 'his', 'how', 'how’s',
    'i', 'i’d', 'i’ll', 'i’m', 'i’ve', 'if', 'in', 'into', 'is', 'isn’t', 'it',
    'it’s', 'its', 'itself', 'let’s', 'me', 'more', 'most', 'mustn’t', 'my',
    'myself', 'no', 'nor', 'not', 'of', 'off', 'on', 'once', 'only', 'or', 'other',
    'ought', 'our', 'ours', 'ourselves', 'out', 'over', 'own', 'same', 'shan’t',
    'she', 'she’d', 'she’ll', 'she’s', 'should', 'shouldn’t', 'so', 'some', 'such',
    'than', 'that', 'that’s', 'the', 'their', 'theirs', 'them', 'themselves', 'then',
    'there', 'there’s', 'these', 'they', 'they’d', 'they’ll', 'they’re', 'they’ve',
    'this', 'those', 'through', 'to', 'too', 'under', 'until', 'up', 'very', 'was',
    'wasn’t', 'we', 'we’d', 'we’ll', 'we’re', 'we’ve', 'were', 'weren’t', 'what',
    'what’s', 'when', 'when’s', 'where', 'where’s', 'which', 'while', 'who', 'who’s',
    'whom', 'why', 'why’s', 'with', 'won’t', 'would', 'wouldn’t', 'you', 'you’d',
    'you’ll', 'you’re', 'you’ve', 'your', 'yours', 'yourself', 'yourselves', 'ain’t',
    'gotta', 'gonna', 'wanna', 'cuz', 'oh', 'uh', 'hmm', 'eh', 'aye', 'nay', 'yea',
    'yes', 'no', 'ok', 'okay', 'uh-huh', 'uh-oh', 'hey', 'hi', 'hello','way','send'
} )

stop_words.update( { '2','4','ur','txt','ltgt','lor','da','wat','ã¼','hey','told','said','u','im','just','like','mr','text','stop','send'
                  'best','number','new','good','got','dont','come','want','year','use','need','day','going','net','tell','later','today','thats'} )

def preprocess_text(text):
    text = text.lower()
    text = text.translate(str.maketrans('', '', string.punctuation))
    tokens = text.split()
    tokens = [word for word in tokens if word not in stop_words and word.isalpha() and len(word)>2]  # remove stopwords
    return tokens


ModuleNotFoundError: No module named 'stopwordsiso'

## *Data Loading and Preprocessing:*

Before we can teach our model anything, we need to prepare the data.

For the BBC News dataset, the articles are stored in different folders inside a ZIP file — each folder representing one topic. We extract each .txt file, read its contents, and record both the text and its category label.

Next, we clean the data:

1. Convert everything to lowercase.

2. Remove punctuation and numbers.

3. Split the text into a list of words (tokens).

We repeat a similar process for the SMS dataset, cleaning and tokenizing each message.
Finally, we divide each dataset into a training set (used for learning) and a test set (used to check performance).

This preprocessing ensures the text is consistent, simple, and ready for the mathematical model to work effectively.

In [ ]:
# Data Loading and Preprocessing of SMS_SPAM_Collection

zip_path_SMS = r"C:\Users\VENKATA SAI KRISHNA\Downloads\sms+spam+collection.zip"
with zipfile.ZipFile(zip_path_SMS) as z_SMS:
    with z_SMS.open(z_SMS.namelist()[0]) as f_SMS:  # open 1st file
        df_SMS = pd.read_csv(f_SMS, sep='\t', names=['Label', 'Message'], encoding='latin-1')
df_SMS['Tokens'] = df_SMS['Message'].apply(preprocess_text)

# Data Loading and Preprocessing of BBC_News_Dataset

zip_path_BBC = r"C:\Users\VENKATA SAI KRISHNA\Downloads\archive.zip"

labels_BBC = []
filenames_BBC = []
texts_BBC = []

with zipfile.ZipFile(zip_path_BBC) as z_BBC:
    for file_name in z_BBC.namelist():
        if file_name.endswith('.txt'):
            # Get the label from the parent folder
            folder_path_BBC = os.path.dirname(file_name)
            doc_label = os.path.basename(folder_path_BBC)  # e.g., 'tech', 'business'

            # Get the file name
            fname = os.path.basename(file_name)

            # Read the file content
            with z_BBC.open(file_name) as f_BBC:
                content = f_BBC.read().decode('utf-8', errors='ignore').strip()

            # Append to lists
            labels_BBC.append(doc_label)
            filenames_BBC.append(fname)
            texts_BBC.append(content)

df_BBC = pd.DataFrame({
    'Label': labels_BBC,    # renamed from 'class' to 'label'
    'Text': texts_BBC
})

df_BBC['Tokens'] = df_BBC['Text'].apply(preprocess_text)


In [ ]:
# Split the data into training and test sets:

df_SMS_train,df_SMS_test=train_test_split_df(df_SMS,0.2,'Label',42)
df_BBC_train,df_BBC_test=train_test_split_df(df_BBC,0.2,'Label',42)


## *Naive Bayes Implementation:*

1. Training phase:
   We go through all documents and count how often each word appears in each class.
   
   From these counts, we calculate two main things:
   
   Prior probability: how common each class is overall.
   
   Likelihood: how likely each word is to appear in that class.

3. Prediction phase:
   
   For a new document, we combine the probabilities of all words in it and choose the class with the highest resulting probability.
   
   Since multiplying many small probabilities can lead to underflow, we use log probabilities instead.
   
This approach gives us a good model that’s easy to debug and understand.

In [ ]:
# Naive Bayes Implementation of SMS_Spam_Collection

labels_SMS = df_SMS['Label'].unique()
doc_counts_SMS = {cls: Counter() for cls in labels_SMS}
docs_in_class_SMS={'spam':len(df_SMS_train[df_SMS_train['Label']=='spam']), 'ham':len(df_SMS_train[df_SMS_train['Label']=='ham'])}
total_docs_SMS=len(df_SMS_train)
Vocabulary_SMS = set()

card_V1=len(Vocabulary_SMS)
alpha_SMS = 0.1

for i in range(len(df_SMS_train)):
    cls_SMS = df_SMS_train.iloc[i]['Label']
    words_SMS = df_SMS_train.iloc[i]['Tokens']
    doc_counts_SMS[cls_SMS].update(set(words_SMS))
    Vocabulary_SMS.update(words_SMS)

top_words_SMS={'spam':doc_counts_SMS['spam'].most_common(), 'ham':doc_counts_SMS['ham'].most_common() }
for cls in top_words_SMS:
    top_words_SMS[cls]= [ word for word in top_words_SMS[cls] if word not in stopwords_iso ]

def prior_SMS(C):
    return docs_in_class_SMS[C]/total_docs_SMS

def bin_ON_OFF_likelihood(w,C):
    return (doc_counts_SMS[C][w]+alpha_SMS)/(docs_in_class_SMS[C]+2*alpha_SMS)

base_log_1_minus_p = {}
log_odds = {C:{} for C in labels_SMS}

for C in labels_SMS:
    s = 0.0
    for w in Vocabulary_SMS:
        p = (doc_counts_SMS[C][w] + alpha_SMS) / (docs_in_class_SMS[C] + 2*alpha_SMS)
        s += np.log(1 - p)
        log_odds[C][w] = np.log(p) - np.log(1 - p)
    base_log_1_minus_p[C] = s

def score_SMS(C,doc):
    s = np.log(prior_SMS(C)) + base_log_1_minus_p[C]
    for w in set(doc):
        if w in Vocabulary_SMS:
            s += log_odds[C][w]
    return s

def predict_class_SMS(doc):
    s=score_SMS('spam',doc)
    h=score_SMS('ham',doc)
    if s>h:
        return 'spam'
    else:
        return 'ham'

# Naive Bayes Implementation of BBC_News_Dataset

labels_BBC = df_BBC['Label'].unique()
word_counts_BBC = {cls: Counter() for cls in labels_BBC}
total_words_BBC = {cls: 0 for cls in labels_BBC}
docs_in_class_BBC= {
    label:len(df_BBC_train[df_BBC_train['Label']==label]) for label in labels_BBC
}
total_docs_BBC=len(df_BBC_train)
Vocabulary_BBC = set()

for i in range(len(df_BBC_train)):
    cls_BBC = df_BBC_train.iloc[i]['Label']
    words_BBC = df_BBC_train.iloc[i]['Tokens']
    word_counts_BBC[cls_BBC].update(words_BBC)
    total_words_BBC[cls_BBC] += len(words_BBC)
    Vocabulary_BBC.update(words_BBC)

card_V2 = len(Vocabulary_BBC)

alpha_BBC = 0.1

top_words_BBC= {'business':word_counts_BBC['business'].most_common(),
                'entertainment':word_counts_BBC['entertainment'].most_common(),
                'politics':word_counts_BBC['politics'].most_common(),
                'sport':word_counts_BBC['sport'].most_common(),
                'tech':word_counts_BBC['tech'].most_common(),
            }
for cls in top_words_BBC:
    top_words_BBC[cls]= [ word for word in top_words_BBC[cls] if word not in stopwords_iso ]

def prior_BBC(C):
    return docs_in_class_BBC[C]/ total_docs_BBC

def multinomial_likelihood(w, C):
    return (word_counts_BBC[C][w] + alpha_BBC) / (total_words_BBC[C] + alpha_BBC * card_V2)

def score_BBC(C, doc):
    scr = np.log(prior_BBC(C))
    for w in doc:
        if w in Vocabulary_BBC:
            scr += np.log(multinomial_likelihood(w, C))
    return scr

def predict_class_BBC(doc):
    scores={ 'business':0, 'entertainment':0, 'tech':0, 'sport':0, 'politics':0 }
    for cls in scores:
        scores[cls]=score_BBC(cls,doc)
    best_class = max(scores, key=scores.get)
    return best_class

## *Evaluation:*

Once the model is trained, we test it on unseen data and measure:

1. Accuracy: the percentage of correct predictions.
    
2. Confusion Matrix: a table showing where the model made correct predictions and where it got confused.
    
For the BBC dataset, accuracy is often quite good (typically above 90%) because the categories use distinct vocabularies.

The confusion matrix provides deeper insight, revealing which classes overlap the most in wording.

In [ ]:
# Evaluation of SMS_Spam_Collection

def accuracy_SMS():
    matrix=[[0,0],[0,0]]
    for i in range(len(df_SMS_test)):
        pred=predict_class_SMS(df_SMS_test.iloc[i]['Tokens'])
        act=df_SMS_test.iloc[i]['Label']
        if act=='ham':
            if pred=='ham':
                matrix[0][0]+=1
            else:
                matrix[0][1]+=1
        else:
            if pred=='ham':
                matrix[1][0]+=1
            else:
                matrix[1][1]+=1
    data=[ ['Actual: ham',matrix[0][0],matrix[0][1]], ['Actual: spam',matrix[1][0],matrix[1][1]] ]
    DF=pd.DataFrame(data,columns=["","Predicted: ham","Predicted: spam"])
    return (matrix[0][0]+matrix[1][1])/len(df_SMS_test),DF

alpha_SMS = 0.1  # Choose alpha_sms here

acc_SMS,DF_SMS=accuracy_SMS()
print("SMS_Spam_Collection :","\n")
print("\t","Accuracy: ",end="")
print(acc_SMS,"\n")
print("\t","Confusion Matrix: ")
print(DF_SMS,"\n")

print("\t","Sample test Cases: ")
print("\t\t","Win free tickets now!!!",":",predict_class_SMS(preprocess_text("Win free tickets now!!!")))
print("\t\t","Are you coming to the meeting?",":",predict_class_SMS(preprocess_text("Are you coming to the meeting?")))
print("\t\t","URGENT! You won $1000",":",predict_class_SMS(preprocess_text("URGENT! You won $1000")))
print("\t\t","See you tomorrow at lunch",":",predict_class_SMS(preprocess_text("See you tomorrow at lunch")),"\n")


# Evaluation of BBC_News_Dataset

def accuracy_BBC():
    matrix=[[0,0,0,0,0] for i in range(5)]
    dic={'business':0,'entertainment':1,'politics':2,'sport':3,'tech':4}
    for i in range(len(df_BBC_test)):
        pred=predict_class_BBC(df_BBC_test.iloc[i]['Tokens'])
        act=df_BBC_test.iloc[i]['Label']
        matrix[dic[act]][dic[pred]]+=1
    data=[
            ['Actual: business',matrix[0][0],matrix[0][1],matrix[0][2],matrix[0][3],matrix[0][4]],
            ['Actual: entertainment',matrix[1][0],matrix[1][1],matrix[1][2],matrix[1][3],matrix[1][4]],
            ['Actual: politics',matrix[2][0],matrix[2][1],matrix[2][2],matrix[2][3],matrix[2][4]],
            ['Actual: sport',matrix[3][0],matrix[3][1],matrix[3][2],matrix[3][3],matrix[3][4]],
            ['Actual: tech',matrix[4][0],matrix[4][1],matrix[4][2],matrix[4][3],matrix[4][4]]
         ]
    DF=pd.DataFrame(data,columns=["","Predicted: business","Predicted: entertainment","Predicted: politics","Predicted: sport","Predicted: tech"])
    return (matrix[0][0]+matrix[1][1]+matrix[2][2]+matrix[3][3]+matrix[4][4])/len(df_BBC_test),DF

alpha_BBC = 0.1 # Choose alpha_bbc here

acc_BBC,DF_BBC=accuracy_BBC()
print("\n","BBC_News_Dataset :","\n")
print("\t","Accurary: ",end="")
print(acc_BBC,"\n")
print("\t","Confusion Matrix: ")
print(DF_BBC,"\n")

print("\t","Sample test Cases: ")
print("\t\t","Stock market crashes as oil prices rise",":",predict_class_BBC(preprocess_text("Stock market crashes as oil prices rise")))
print("\t\t","Premier League team wins the championship",":",predict_class_BBC(preprocess_text("Premier League team wins the championship")))
print("\t\t","Government passes new healthcare reform",":",predict_class_BBC(preprocess_text("Government passes new healthcare reform")))
print("\t\t","Apple releases latest iPhone with new features",":",predict_class_BBC(preprocess_text("Apple releases latest iPhone with new features")))
print("\t\t","Celebrity announces new film project",":",predict_class_BBC(preprocess_text("Celebrity announces new film project")))


SMS_Spam_Collection : 

	 Accuracy: 0.9829443447037702 

	 Confusion Matrix: 
                 Predicted: ham  Predicted: spam
0   Actual: ham             964                1
1  Actual: spam              18              131 

	 Sample test Cases: 
		 Win free tickets now!!! : spam
		 Are you coming to the meeting? : ham
		 URGENT! You won $1000 : spam
		 See you tomorrow at lunch : ham 


 BBC_News_Dataset : 

	 Accurary: 0.996060776589758 

	 Confusion Matrix: 
                          Predicted: business  Predicted: entertainment  \
0       Actual: business                  406                         0   
1  Actual: entertainment                    0                       306   
2       Actual: politics                    0                         0   
3          Actual: sport                    2                         0   
4           Actual: tech                    0                         1   

   Predicted: politics  Predicted: sport  Predicted: tech  
0                    

## *Analysis:*

To understand what the model has learned, we can inspect the words that are most characteristic of each class.

For example:

In the sport category, words like “match”, “team”, and “goal” might dominate.

In tech, words such as “software”, “computer”, and “internet” may appear frequently.

For SMS spam, typical strong indicators include “free”, “win”, “urgent”, and “offer”.

This analysis not only confirms that the model is learning meaningful distinctions but also helps us interpret and trust its predictions.

In [ ]:
# Analysis of SMS_Spam_Collection

print("Top 10 indicative words of spam :",[ top_words_SMS['spam'][_][0] for _ in range(10) ])
print("Top 10 indicative words of ham :",[ top_words_SMS['ham'][_][0] for _ in range(10) ])
print("\n")

# Analysis of BBC_News_Dataset

print("Top 10 indicative words of business :",[ top_words_BBC['business'][_][0] for _ in range(10) ])
print("Top 10 indicative words of entertainment :",[ top_words_BBC['entertainment'][_][0] for _ in range(10) ])
print("Top 10 indicative words of politics :",[ top_words_BBC['politics'][_][0] for _ in range(10) ])
print("Top 10 indicative words of sport :",[ top_words_BBC['sport'][_][0] for _ in range(10) ])
print("Top 10 indicative words of tech :",[ top_words_BBC['tech'][_][0] for _ in range(10) ])


Top 10 indicative words of spam : ['free', 'mobile', 'claim', 'reply', 'prize', 'urgent', 'won', 'cash', 'win', 'service']
Top 10 indicative words of ham : ['know', 'ill', 'time', 'home', 'love', 'sorry', 'think', 'night', 'hope', 'work']


Top 10 indicative words of business : ['market', 'growth', 'company', 'economy', 'bank', 'firm', 'economic', 'oil', 'sales', 'government']
Top 10 indicative words of entertainment : ['film', 'best', 'music', 'years', 'awards', 'won', 'award', 'band', 'director', 'star']
Top 10 indicative words of politics : ['government', 'labour', 'people', 'election', 'blair', 'party', 'minister', 'brown', 'howard', 'public']
Top 10 indicative words of sport : ['game', 'win', 'world', 'england', 'time', 'players', 'cup', 'play', 'wales', 'second']
Top 10 indicative words of tech : ['people', 'mobile', 'technology', 'games', 'music', 'users', 'digital', 'software', 'game', 'phone']


## *Conclusion:*

This project helped us understand how text classification works using the Naive Bayes algorithm.

We learned that:

1. Cleaning and preparing the text is very important before training a model.

2. Even a simple model like Naive Bayes can give good accuracy on real data.

3. The model uses word frequencies to decide which class a text belongs to.

4. The confusion matrix helps us see where the model gets confused.

5. Looking at top words for each class shows what words best represent that topic.

Overall, we saw that with some basic math and clear preprocessing, we can build a model that understands and classifies text quite well — without using any complex tools.